In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
count = 0
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
        count += 1
        if count == 10:
            break
    if count == 10:
        break
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/gan-getting-started/monet_jpg/f4413e97bd.jpg
/kaggle/input/competitions/gan-getting-started/monet_jpg/7341d96c1d.jpg
/kaggle/input/competitions/gan-getting-started/monet_jpg/de6f71b00f.jpg
/kaggle/input/competitions/gan-getting-started/monet_jpg/99d94af5dd.jpg
/kaggle/input/competitions/gan-getting-started/monet_jpg/99a51d3e25.jpg
/kaggle/input/competitions/gan-getting-started/monet_jpg/d05cab011d.jpg
/kaggle/input/competitions/gan-getting-started/monet_jpg/4e05523825.jpg
/kaggle/input/competitions/gan-getting-started/monet_jpg/c68c52e8fc.jpg
/kaggle/input/competitions/gan-getting-started/monet_jpg/40d7d18ad3.jpg
/kaggle/input/competitions/gan-getting-started/monet_jpg/f96a8de9f3.jpg


In [2]:
from glob import glob
import os
import matplotlib

matplotlib.use('Agg')

# Try standard path first, fall back to competitions path
base = '/kaggle/input/gan-getting-started'
if not os.path.exists(base):
    base = '/kaggle/input/competitions/gan-getting-started'

monet_paths = glob(f'{base}/monet_jpg/*.jpg')
photo_paths = glob(f'{base}/photo_jpg/*.jpg')

print("Monet images:", len(monet_paths))
print("Photo images:", len(photo_paths))

Monet images: 300
Photo images: 7038


Lets see what kind of images are there in the given data

In [3]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

os.makedirs('/kaggle/tmp', exist_ok=True)

img1 = mpimg.imread(monet_paths[0])
img2 = mpimg.imread(photo_paths[0])

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(img1)
plt.title("Monet")

plt.subplot(1,2,2)
plt.imshow(img2)
plt.title("Photo")

plt.savefig('/kaggle/tmp/sample.png')
plt.close()

In [4]:
import tensorflow as tf

IMG_SIZE = 256

def load_image(path):
    img = tf.io.read_file(path)                  # read jpg file
    img = tf.image.decode_jpeg(img, channels=3)  # convert to pixels
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = (tf.cast(img, tf.float32) / 127.5) - 1 # normalize to [-1,1]
    return img

monet_tensor = load_image(monet_paths[0])
photo_tensor = load_image(photo_paths[0])

print("Monet tensor shape:", monet_tensor.shape)
print("Min/Max:", tf.reduce_min(monet_tensor).numpy(),
      tf.reduce_max(monet_tensor).numpy())

2026-06-08 10:28:23.630431: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780914503.851656      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780914503.916271      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780914504.446099      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780914504.446151      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780914504.446154      23 computation_placer.cc:177] computation placer alr

Monet tensor shape: (256, 256, 3)
Min/Max: -1.0 1.0


I0000 00:00:1780914531.768703      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


In [5]:
BATCH_SIZE = 1

monet_ds = (
    tf.data.Dataset.from_tensor_slices(monet_paths)
    .shuffle(len(monet_paths))
    .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
)

photo_ds = (
    tf.data.Dataset.from_tensor_slices(photo_paths)
    .shuffle(len(photo_paths))
    .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
)

sample_batch = next(iter(monet_ds))

print(sample_batch.shape)

(1, 256, 256, 3)


In [6]:
def residual_block(x):
    y = tf.keras.layers.Conv2D(256, 3, padding='same')(x)
    y = tf.keras.layers.ReLU()(y)
    y = tf.keras.layers.Conv2D(256, 3, padding='same')(y)

    return tf.keras.layers.Add()([x, y])

Now we meet the generator for monet

In [7]:
def build_generator():
    inputs = tf.keras.Input(shape=(256, 256, 3))

    # Encoder
    x = tf.keras.layers.Conv2D(64, 7, padding='same')(inputs)
    x = tf.keras.layers.ReLU()(x)

    x = tf.keras.layers.Conv2D(128, 3, strides=2, padding='same')(x)
    x = tf.keras.layers.ReLU()(x)

    x = tf.keras.layers.Conv2D(256, 3, strides=2, padding='same')(x)
    x = tf.keras.layers.ReLU()(x)

    # Residual blocks (style transformation core)
    for _ in range(6):
        x = residual_block(x)

    # Decoder (upsampling back to image size)
    x = tf.keras.layers.Conv2DTranspose(128, 3, strides=2, padding='same')(x)
    x = tf.keras.layers.ReLU()(x)

    x = tf.keras.layers.Conv2DTranspose(64, 3, strides=2, padding='same')(x)
    x = tf.keras.layers.ReLU()(x)

    # Output layer
    outputs = tf.keras.layers.Conv2D(3, 7, padding='same', activation='tanh')(x)

    return tf.keras.Model(inputs, outputs)

In [8]:
def build_discriminator():
    inputs = tf.keras.Input(shape=(256, 256, 3))

    x = tf.keras.layers.Conv2D(64, 4, strides=2, padding='same')(inputs)
    x = tf.keras.layers.LeakyReLU(0.2)(x)

    x = tf.keras.layers.Conv2D(128, 4, strides=2, padding='same')(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)

    x = tf.keras.layers.Conv2D(256, 4, strides=2, padding='same')(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)

    x = tf.keras.layers.Conv2D(512, 4, strides=1, padding='same')(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)

    # Patch output (NOT single value)
    outputs = tf.keras.layers.Conv2D(1, 4, padding='same')(x)

    return tf.keras.Model(inputs, outputs)

In [9]:
loss_obj = tf.keras.losses.BinaryCrossentropy(
    from_logits=True)

def discriminator_loss(real, fake):
    real_loss = loss_obj(
        tf.ones_like(real), real)

    fake_loss = loss_obj(
        tf.zeros_like(fake), fake)

    return real_loss + fake_loss

def generator_loss(fake):
    return loss_obj(
        tf.ones_like(fake), fake)

In [10]:
LAMBDA = 10

def cycle_loss(real, cycled):
    return LAMBDA * tf.reduce_mean(
        tf.abs(real - cycled))

In [11]:
def identity_loss(real, same):
    return LAMBDA * 0.5 * tf.reduce_mean(
        tf.abs(real - same))

| Optimizer                 | Updates    |
| ------------------------- | ---------- |
| generator_g_optimizer     | G weights  |
| generator_f_optimizer     | F weights  |
| discriminator_x_optimizer | Dx weights |
| discriminator_y_optimizer | Dy weights |

In [12]:
generator_g_optimizer = tf.keras.optimizers.Adam(
    2e-4, beta_1=0.5)

generator_f_optimizer = tf.keras.optimizers.Adam(
    2e-4, beta_1=0.5)

discriminator_x_optimizer = tf.keras.optimizers.Adam(
    2e-4, beta_1=0.5)

discriminator_y_optimizer = tf.keras.optimizers.Adam(
    2e-4, beta_1=0.5)

In [13]:
generator_g = build_generator()   # Photo -> Monet
generator_f = build_generator()   # Monet -> Photo

discriminator_x = build_discriminator()  # Photo critic
discriminator_y = build_discriminator()  # Monet critic

In [14]:
@tf.function
def train_step(real_x, real_y):

    with tf.GradientTape(persistent=True) as tape:

        # Generate images
        fake_y = generator_g(real_x, training=True)
        fake_x = generator_f(real_y, training=True)

        # Cycle reconstruction
        cycled_x = generator_f(fake_y, training=True)
        cycled_y = generator_g(fake_x, training=True)

        # Identity
        same_x = generator_f(real_x, training=True)
        same_y = generator_g(real_y, training=True)

        # Discriminator predictions
        disc_real_x = discriminator_x(real_x, training=True)
        disc_real_y = discriminator_y(real_y, training=True)

        disc_fake_x = discriminator_x(fake_x, training=True)
        disc_fake_y = discriminator_y(fake_y, training=True)

        # Generator adversarial losses
        gen_g_loss = generator_loss(disc_fake_y)
        gen_f_loss = generator_loss(disc_fake_x)

        # Cycle loss
        total_cycle_loss = (
            cycle_loss(real_x, cycled_x) +
            cycle_loss(real_y, cycled_y)
        )

        # Total generator loss
        total_gen_g_loss = (
            gen_g_loss
            + total_cycle_loss
            + identity_loss(real_y, same_y)
        )

        total_gen_f_loss = (
            gen_f_loss
            + total_cycle_loss
            + identity_loss(real_x, same_x)
        )

        # Discriminator loss
        disc_x_loss = discriminator_loss(
            disc_real_x, disc_fake_x)

        disc_y_loss = discriminator_loss(
            disc_real_y, disc_fake_y)

    # Compute gradients
    generator_g_gradients = tape.gradient(
        total_gen_g_loss,
        generator_g.trainable_variables)

    generator_f_gradients = tape.gradient(
        total_gen_f_loss,
        generator_f.trainable_variables)

    discriminator_x_gradients = tape.gradient(
        disc_x_loss,
        discriminator_x.trainable_variables)

    discriminator_y_gradients = tape.gradient(
        disc_y_loss,
        discriminator_y.trainable_variables)

    # Apply gradients
    generator_g_optimizer.apply_gradients(
        zip(generator_g_gradients,
            generator_g.trainable_variables))

    generator_f_optimizer.apply_gradients(
        zip(generator_f_gradients,
            generator_f.trainable_variables))

    discriminator_x_optimizer.apply_gradients(
        zip(discriminator_x_gradients,
            discriminator_x.trainable_variables))

    discriminator_y_optimizer.apply_gradients(
        zip(discriminator_y_gradients,
            discriminator_y.trainable_variables))

In [15]:
EPOCHS = 15

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    step = 0
    for image_x, image_y in tf.data.Dataset.zip((photo_ds, monet_ds)):
        train_step(image_x, image_y)

        step += 1
        if step % 100 == 0:
            print(f"Step {step}")

    print("Done")


Epoch 1/15


I0000 00:00:1780914547.167034      66 cuda_dnn.cc:529] Loaded cuDNN version 91002


Step 100
Step 200
Step 300
Done

Epoch 2/15
Step 100
Step 200
Step 300
Done

Epoch 3/15
Step 100
Step 200
Step 300
Done

Epoch 4/15
Step 100
Step 200
Step 300
Done

Epoch 5/15
Step 100
Step 200
Step 300
Done

Epoch 6/15
Step 100
Step 200
Step 300
Done

Epoch 7/15
Step 100
Step 200
Step 300
Done

Epoch 8/15
Step 100
Step 200
Step 300
Done

Epoch 9/15
Step 100
Step 200
Step 300
Done

Epoch 10/15
Step 100
Step 200
Step 300
Done

Epoch 11/15
Step 100
Step 200
Step 300
Done

Epoch 12/15
Step 100
Step 200
Step 300
Done

Epoch 13/15
Step 100
Step 200
Step 300
Done

Epoch 14/15
Step 100
Step 200
Step 300
Done

Epoch 15/15
Step 100
Step 200
Step 300
Done


In [16]:
import matplotlib.pyplot as plt

# Use a fresh dataset for visualization, not photo_ds
viz_ds = (
    tf.data.Dataset.from_tensor_slices(photo_paths[:3])
    .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(1)
)

for img_batch in viz_ds:
    prediction = generator_g(img_batch, training=False)
    original = (img_batch[0].numpy() + 1) / 2
    generated = (prediction[0].numpy() + 1) / 2

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(original)
    axes[0].set_title("Original Photo")
    axes[0].axis("off")
    axes[1].imshow(generated)
    axes[1].set_title("Generated Monet")
    axes[1].axis("off")
    plt.savefig('/kaggle/tmp/preview.png')  # save instead of show
    plt.close()

In [17]:
import zipfile
import io
import numpy as np
from PIL import Image
from tqdm import tqdm

photo_ds_infer = (
    tf.data.Dataset.from_tensor_slices(photo_paths)
    .map(lambda path: (path, load_image(path)), num_parallel_calls=tf.data.AUTOTUNE)
    .batch(1)
)

with zipfile.ZipFile("images.zip", "w") as img_zip:
    for path_batch, img_batch in tqdm(photo_ds_infer.take(10000)):
        prediction = generator_g(img_batch, training=False)
        img = prediction[0].numpy()
        img = ((img + 1) * 127.5).astype(np.uint8)
        img_pil = Image.fromarray(img)
        buffer = io.BytesIO()
        img_pil.save(buffer, format="JPEG")
        
        fname = os.path.basename(path_batch[0].numpy().decode())
        img_zip.writestr(fname, buffer.getvalue())


print("images.zip created!")

100%|██████████| 7038/7038 [04:20<00:00, 27.04it/s]

images.zip created!


In [18]:
import zipfile

with zipfile.ZipFile("images.zip", "r") as z:
    print("Image count:", len(z.namelist()))
    print("First few:", z.namelist()[:5])

Image count: 7038
First few: ['fb97febc5f.jpg', 'c54c5368af.jpg', '4a06596662.jpg', '2629524a69.jpg', '5e64b64de8.jpg']
